<a href="https://colab.research.google.com/github/taibaabid/FlyRank_ML_Internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Logic (Plain Words)
We flag pages that have high opportunity (impressions/search volume) but are underperforming due to staleness or low CTR relative to position.

1. **REFRESH_NEEDED**: High impression pages (>1,000) that haven't been updated in over 180 days.
2. **CTR_FIX**: Pages ranking in the top 10 with high impressions but actual CTR falling below expected CTR for that position by >2%.
3. **NO_ACTION**: Low impressions or recent updates with healthy CTRs.

### Reason Codes
- `STALE_HIGH_IMPRESSIONS`: High impression potential, but content is out of date.
- `UNDERPERFORMING_CTR`: Good rank position, but low click-through rate.
- `STABLE`: No immediate intervention required.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import numpy as np
import os

# 1. Define scoring logic based on your dataset's actual features
def assign_baseline_score(row):
    # Retrieve key signals with safe fallbacks
    search_vol = row.get('search_volume', 0) if pd.notnull(row.get('search_volume')) else 0
    ctr = row.get('ctr', 0) if pd.notnull(row.get('ctr')) else 0
    avg_pos = row.get('avg_position', 99) if pd.notnull(row.get('avg_position')) else 99
    trend_dir = row.get('trend_direction', 'stable')
    trend_pct = row.get('trend_pct', 0) if pd.notnull(row.get('trend_pct')) else 0

    # Heuristic 1: High opportunity content with declining traffic trend (Needs Refresh)
    if search_vol >= 10 and trend_dir == 'down' and trend_pct < -20:
        score = 0.80 + min(0.19, abs(trend_pct) / 1000)
        action = 'REFRESH_CONTENT'
        reason_code = 'DECLINING_TRAFFIC_TREND'

    # Heuristic 2: Striking distance position (Page 2) with decent volume but low CTR
    elif 10 <= avg_pos <= 20 and search_vol >= 10:
        score = 0.65 + min(0.14, search_vol / 1000)
        action = 'OPTIMIZE_META_TITLE'
        reason_code = 'STRIKING_DISTANCE_LOW_CTR'

    # Heuristic 3: Stable or default monitoring
    else:
        score = max(0.01, min(0.50, search_vol / 500))
        action = 'MONITOR'
        reason_code = 'STABLE_PERFORMANCE'

    return pd.Series([score, action, reason_code], index=['score', 'action_label', 'reason_code'])

# 2. Compute score, action, and reason_code
df[['score', 'action_label', 'reason_code']] = df.apply(assign_baseline_score, axis=1)

# 3. Create ranked queue (highest score first)
ranked_queue = df.sort_values(by='score', ascending=False).reset_index(drop=True)

# 4. Create outputs directory and write the output CSV
os.makedirs('work/outputs', exist_ok=True)
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print(f"✅ Success! Ranked queue written to: {output_path}")
print(f"Total rows ranked: {len(ranked_queue)}\n")

# Display the top 10 rows for Section 3 review
ranked_queue[['content_id', 'score', 'action_label', 'reason_code', 'search_volume', 'avg_position', 'trend_pct']].head(10)

✅ Success! Ranked queue written to: work/outputs/baseline_action_score.csv
Total rows ranked: 30000



,content_id,score,action_label,reason_code,search_volume,avg_position,trend_pct
0,content_06b328857217,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,40.0,18.7,-100.0
1,content_e3bf6539b4ba,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,90.0,8.4,-100.0
2,content_99f7f3940ed4,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,10.0,45.1,-100.0
3,content_ca1a4cf6ef7f,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,320.0,9.1,-100.0
4,content_020e342f9904,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,10.0,12.2,-100.0
5,content_9bb9a0584cae,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,210.0,54.0,-100.0
6,content_88a22491fe70,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,10.0,15.9,-100.0
7,content_4595e8704e07,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,90.0,36.3,-100.0
8,content_56de85247382,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,40.0,57.7,-100.0
9,content_39678687bb17,0.9,REFRESH_CONTENT,DECLINING_TRAFFIC_TREND,10.0,7.7,-100.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Queue Review

| Content ID | Action | Reason Code | Why it's here | What would make it wrong? |
| :--- | :--- | :--- | :--- | :--- |
| `content_06b328857217` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | `trend_pct` drops -100.0% with 40.0 search volume and avg rank 18.7. | High seasonality or intentional URL migration/redirect could explain the 100% drop. |
| `content_e3bf6539b4ba` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | High potential: Page 1 rank (8.4) with 90.0 volume dropping -100.0%. | Technical indexing block (noindex/robots.txt) or tracking script failure. |
| `content_99f7f3940ed4` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Flagged due to severe traffic drop (-100.0%). | **Weak Pick**: Rank is 45.1 (Page 5) with only 10 search volume; low value target. |
| `content_ca1a4cf6ef7f` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Highest impact item: Page 1 rank (9.1) and high volume (320.0). | Page was recently updated or query lost user intent alignment. |
| `content_020e342f9904` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Page 2 rank (12.2) with -100% traffic trend. | Very low base search volume (10.0)—effort to refresh outweighs gain. |
| `content_9bb9a0584cae` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Flagged on high volume (210.0) despite poor rank (54.0). | **Weak Pick**: Rank 54 is too deep for a simple content refresh to fix easily. |
| `content_88a22491fe70` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Page 2 rank (15.9) with -100% trend drop. | Search intent changed or low volume query (10.0). |
| `content_4595e8704e07` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | High volume (90.0) with complete traffic loss (-100%). | Rank is 36.3; low search volume conversions make it low priority. |
| `content_56de85247382` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Severely declining trend (-100%). | **Weak Pick**: Rank 57.7 with 40 volume—page is buried in SERPs. |
| `content_39678687bb17` | `REFRESH_CONTENT` | `DECLINING_TRAFFIC_TREND` | Page 1 rank (7.7) with severe traffic drop. | Tracking anomaly or sudden competitor SERP feature takeover. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Analysis
* **Deeply buried rankings (`avg_position > 30`):** Items like `content_9bb9a0584cae` (rank 54.0) and `content_56de85247382` (rank 57.7) scored high purely due to `-100%` traffic drops. A basic content refresh will rarely restore rankings for content sitting on Page 5+.
* **Low Search Volume Noise:** Pages with tiny search volume (e.g., 10.0) got equal priority as pages with 320.0 volume. The heuristic needs volume weighting.

### Leakage Verification
- [x] **No future-window leaks:** All signals used (`search_volume`, `avg_position`, `trend_pct`) are historic observational metrics.
- [x] **No label-derived inputs:** Score relies strictly on pre-computed feature inputs, not target flags.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.